# CS2 EXP-6 — NeoBERT-250M Frozen Encoder Linear Probe

## 1. Dependencies

In [1]:
import sys
import subprocess

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "--extra-index-url", "https://download.pytorch.org/whl/cu121",
    "torch==2.5.1",
    "torchvision",
    "torchaudio",
    "transformers<4.49.0",  # Avoids the CVE check enforcing PyTorch 2.6
    "numpy<2.1.0",
    "pandas",
    "scikit-learn",
    "matplotlib",
    "pyarrow",
    "joblib",
    "tqdm",
    "psutil",
    "einops",   # NeoBERT (chandar-lab/NeoBERT) remote modeling code dependency
], check=True)

# EXP-6 only needs the frozen NeoBERT encoder (no LoRA/PEFT), so unlike EXP-7
# this doesn't need `peft`/`accelerate`. It still needs `xformers`, pinned to
# match torch==2.5.1 for the same reason documented in EXP-7's install cell:
# an unpinned `-U xformers` silently upgrades torch out from under this pin.
# We force `use_unpadding=False` in case_study_2/models.py (see PDD sec. 5.2),
# so flash-attention is NOT required.
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-deps",
    "xformers==0.0.28.post3",
], check=True)

import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
print("Dependencies installed successfully for CUDA 12.1 driver!")


Dependencies installed successfully for CUDA 12.1 driver!


In [2]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi produced no stdout")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stderr)

import os
print("CUDA_VISIBLE_DEVICES before override:", repr(os.environ.get("CUDA_VISIBLE_DEVICES")))

import torch
print("torch.__version__:", torch.__version__)
print("torch.version.cuda:", torch.version.cuda)
print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch.cuda.device_count():", torch.cuda.device_count())
print("torch already CUDA-initialized:", torch.cuda.is_initialized())


Tue Aug 11 19:05:02 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.288.01             Driver Version: 535.288.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  | 00000000:CD:00.0 Off |                    0 |
| N/A   49C    P0              66W / 500W |      3MiB / 81920MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

In [3]:
import torch
try:
    torch.cuda.init()
except Exception as e:
    print(repr(e))


In [4]:
import subprocess, sys

# Fallback: only needed if the cell above shows torch.cuda.is_available()==False
# after the pinned install (seen occasionally on some CUDA 12.1 hosts). Re-run
# this, then RESTART THE KERNEL, then re-run from the top -- do not just
# continue in the same process, torch's CUDA init is one-shot per process.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "torch", "torchvision", "torchaudio",
    "--index-url", "https://download.pytorch.org/whl/cu121",
], check=True)

print("Reinstalled. Restart the kernel now, then re-run your CUDA check cell.")


Found existing installation: torch 2.5.1+cu121
Uninstalling torch-2.5.1+cu121:
  Successfully uninstalled torch-2.5.1+cu121
Found existing installation: torchvision 0.20.1+cu121
Uninstalling torchvision-0.20.1+cu121:
  Successfully uninstalled torchvision-0.20.1+cu121
Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0


Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp310-cp310-linux_x86_64.whl (780.4 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp310-cp310-linux_x86_64.whl (7.3 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 35.5 MB/s eta 0:00:00


Reinstalled. Restart the kernel now, then re-run your CUDA check cell.


In [5]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch

assert torch.cuda.is_available(), "CUDA is not available!"
assert torch.cuda.device_count() == 1, f"Expected 1 GPU, but PyTorch sees {torch.cuda.device_count()}"

DEVICE = "cuda:0"

print(f"Locked to single GPU: {torch.cuda.get_device_name(0)}")
print(f"Total Visible GPUs in PyTorch: {torch.cuda.device_count()}")


Locked to single GPU: NVIDIA A100-SXM4-80GB
Total Visible GPUs in PyTorch: 1


## 1.5 Settings

In [ ]:
from pathlib import Path
import os

REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
REPO_BRANCH = "simo"

WORKSPACE_ROOT = Path.cwd()

REPO_ROOT = WORKSPACE_ROOT / "DiverseVul--IS-Project"
PROJECT_DIR = REPO_ROOT / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"

DATA_ROOT = WORKSPACE_ROOT / "IntelligentSystemProject" / "VulnerabilityDetectionData"

PROCESSED_DIR = DATA_ROOT / "processed"
MANIFEST_ROOT = DATA_ROOT / "manifests"
OUTPUT_ROOT = DATA_ROOT / "outputs"

SPLIT_ID = "cs1_project_holdout20_innercv_v1"

CODE_COLUMN = "normalized_code"
# CODE_COLUMN = "abstracted_code_v1"
CODE_COLUMN_TAG = "abstracted" if CODE_COLUMN == "abstracted_code_v1" else "normalized"

NORMALIZED_PARQUET = (
    PROCESSED_DIR
    / "rdiversevul_cs1_normalized_plus_abstracted_v2.parquet"
)

OUTER_MANIFEST_PATH = (
    MANIFEST_ROOT
    / SPLIT_ID
    / "outer_holdout"
    / "cs1_outer_project_holdout_manifest.parquet"
)

INNER_MANIFEST_PATH = (
    MANIFEST_ROOT
    / SPLIT_ID
    / "inner_cv"
    / "cs1_project_grouped_5fold_manifest.parquet"
)

EXP6_OUTPUT_DIR = (
    OUTPUT_ROOT
    / "case_study_2"
    / f"exp6_neobert_linear_probe_v1_{CODE_COLUMN_TAG}"
)
EXP6_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_CACHE_DIR = DATA_ROOT / "embedding_cache" / "neobert_v1"
EMBEDDING_CACHE_DIR.mkdir(parents=True, exist_ok=True)

HF_CACHE_DIR = WORKSPACE_ROOT / "IntelligentSystemProject" / "hf_cache"
# os.environ["HF_TOKEN"] = "secret"

C_GRID = (1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0)

RUN_SMOKE_TEST = True
RUN_PROFILE = True
RUN_NESTED_OFFICIAL = True
RUN_CANONICAL_RETRAIN = True
RUN_HOLDOUT_EVAL = True

print("Settings loaded.")
print(f"Workspace: {WORKSPACE_ROOT}")
print(f"Repository: {REPO_ROOT}")
print(f"Data root: {DATA_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")
print(f"Hugging Face cache: {HF_CACHE_DIR}")
print(f"Device: {DEVICE}")


## 2. Clone the repository

In [7]:
import urllib.request
import zipfile
from pathlib import Path

if not REPO_ROOT.exists():
    print(f"Downloading repository (branch: {REPO_BRANCH}) without git...")

    clean_url = REPO_URL.removesuffix(".git")
    zip_url = f"{clean_url}/archive/refs/heads/{REPO_BRANCH}.zip"
    zip_path = Path.cwd() / "repo_temp.zip"

    urllib.request.urlretrieve(zip_url, zip_path)

    print("Extracting files...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(Path.cwd())

    repo_name = clean_url.split("/")[-1]
    extracted_folder = Path.cwd() / f"{repo_name}-{REPO_BRANCH}"
    if extracted_folder.exists():
        extracted_folder.rename(REPO_ROOT)

    zip_path.unlink()
    print("Repository setup complete!")
else:
    print(f"Repository already exists at {REPO_ROOT}")


Extracting files...
Repository setup complete!


## 3. Verify GPU, RAM, and storage budget

In [8]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
assert torch.cuda.is_available(), "CUDA is not available!"
assert torch.cuda.device_count() == 1, f"Expected 1 GPU, but PyTorch sees {torch.cuda.device_count()}"
DEVICE = "cuda:0"
print(f"Locked to single GPU: {torch.cuda.get_device_name(0)}")
print(f"bfloat16 supported: {torch.cuda.is_bf16_supported()}")
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


Locked to single GPU: NVIDIA A100-SXM4-80GB
bfloat16 supported: True
Total VRAM: 84.99 GB


## 4. Data availability check

In [9]:
STORAGE_CAP_GB = 60

required_data_files = {
    "normalized parquet": NORMALIZED_PARQUET,
    "outer holdout manifest": OUTER_MANIFEST_PATH,
    "inner CV manifest": INNER_MANIFEST_PATH,
}

missing = {name: path for name, path in required_data_files.items() if not path.is_file()}

if missing:
    print("Missing required data files:")
    for name, path in missing.items():
        print(f"  - {name}: {path}")
    print(
        "\nThese files are produced by the Case Study 1 pipeline (normalization_v3.py + "
        "split_manifest.py) and were previously synced through Google Drive. On this machine, "
        "either:\n"
        "  1) Copy them from your previous Drive/Colab run into the paths above; or\n"
        "  2) Re-run the Case Study 1 notebook/pipeline against `data/raw/rdiversevul.json` to "
        "regenerate them locally.\n"
        f"Keep an eye on the {STORAGE_CAP_GB} GB storage cap while doing either."
    )
    raise FileNotFoundError("Required processed data/manifests are missing; see instructions above.")
else:
    for name, path in required_data_files.items():
        size_mb = path.stat().st_size / 1e6
        print(f"Found {name}: {path} ({size_mb:.1f} MB)")


Found normalized parquet: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/processed/rdiversevul_cs1_normalized_plus_abstracted_v1.parquet (210.7 MB)
Found outer holdout manifest: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/outer_holdout/cs1_outer_project_holdout_manifest.parquet (1.4 MB)
Found inner CV manifest: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/inner_cv/cs1_project_grouped_5fold_manifest.parquet (1.2 MB)


## 5. Write case_study_2 source files

Writes `models.py` (already NeoBERT-aware, unchanged from EXP-7), `data_loader.py`, and `exp6/exp6_linear_probe.py` into the cloned checkout.

In [10]:
(SRC_DIR / "case_study_2/__init__.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/__init__.py").write_text('')
print("Wrote", "case_study_2/__init__.py")


Wrote case_study_2/__init__.py


In [11]:
(SRC_DIR / "case_study_2/models.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/models.py").write_text('from __future__ import annotations\n\nimport os\nimport warnings\nfrom pathlib import Path\nfrom typing import Optional, Dict, Any, List\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom transformers import AutoConfig, AutoModel, AutoTokenizer\n\n\nDEFAULT_CODE_MODEL = "huggingface/CodeBERTa-small-v1"\nDEFAULT_CODE_TOKENIZER = "huggingface/CodeBERTa-small-v1"\n\n# EXP-7: NeoBERT-250M backbone (Chandar Research Lab). Plug-and-play replacement\n# for the CodeBERTa backbone above -- same hidden size (768), but a 4,096-token\n# RoPE/YaRE context window, SwiGLU activations, and Pre-RMSNorm. Ships as\n# `trust_remote_code` model code on the Hub rather than a native `transformers`\n# architecture, so it needs a couple of extra loading safeguards (see\n# `_neobert_loading_overrides` below).\nDEFAULT_NEOBERT_MODEL = "chandar-lab/NeoBERT"\nDEFAULT_NEOBERT_TOKENIZER = "chandar-lab/NeoBERT"\n\n# Model-name substrings that identify a NeoBERT-family checkpoint and therefore\n# require `trust_remote_code=True` plus the safeguards below. Kept as a simple\n# substring match (rather than a fixed set) so forks/finetunes of NeoBERT\n# (e.g. "chandar-lab/NeoBERT", "someuser/NeoBERT-finetuned-...") are still\n# picked up automatically.\n_NEOBERT_NAME_HINTS = ("neobert",)\n\n\ndef _is_neobert_model(model_name: str) -> bool:\n    name = (model_name or "").lower()\n    return any(hint in name for hint in _NEOBERT_NAME_HINTS)\n\n\ndef configure_huggingface_cache(hf_cache_dir: Optional[str] = None) -> None:\n    if hf_cache_dir:\n        hf_cache_dir = str(hf_cache_dir)\n        os.environ.setdefault("HF_HOME", hf_cache_dir)\n        os.environ.setdefault("HUGGINGFACE_HUB_CACHE", str(Path(hf_cache_dir) / "hub"))\n    os.environ.setdefault("HF_HUB_DISABLE_XET", "1")\n    os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")\n    os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")\n    os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "120")\n    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\n\n\ndef _dtype_from_policy(dtype_policy: str, device: str) -> Optional[torch.dtype]:\n    dtype_policy = (dtype_policy or "auto").lower()\n    device = str(device)\n    if dtype_policy == "float16":\n        return torch.float16 if device == "cuda" else torch.float32\n    if dtype_policy == "bfloat16":\n        return torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else torch.float32\n    if dtype_policy == "float32":\n        return torch.float32\n    if dtype_policy == "auto":\n        if device == "cuda" and torch.cuda.is_bf16_supported():\n            return torch.bfloat16\n        if device == "cuda":\n            return torch.float32\n        return torch.float32\n    raise ValueError(f"Unknown dtype_policy: {dtype_policy}")\n\n\ndef _apply_neobert_config_overrides(config: Any) -> Any:\n    """\n    Mandatory Track-B safeguard (PDD sec. 5.2, GitHub Issue #7 on\n    chandar-lab/NeoBERT -- "How is unpadding handled when unpacking?"):\n    sequence-packing / unpadding in the reference NeoBERT code can leak\n    attention across the pad boundary when the collator doesn\'t also emit\n    packed cu_seqlens, which is exactly our setup (we pad batches instead of\n    packing them). Forcing `use_unpadding=False` makes NeoBERT fall back to\n    strict padded multi-head attention with the HF attention mask, which is\n    the safe/correct path for this pipeline.\n\n    The exact attribute name has moved around across NeoBERT code revisions,\n    so we try the known aliases and only set whichever is actually present on\n    this revision\'s config, rather than hard-failing.\n    """\n    candidate_flags = ("use_unpadding", "unpad_inputs", "unpad", "pack_sequences")\n    matched = False\n    for flag in candidate_flags:\n        if hasattr(config, flag):\n            setattr(config, flag, False)\n            matched = True\n    if not matched:\n        warnings.warn(\n            "[models] Could not find a known unpadding flag on the NeoBERT config "\n            "(checked: %s). This revision of chandar-lab/NeoBERT may handle "\n            "padding differently -- double check attention-mask correctness "\n            "manually (see PDD sec. 5.2 / GitHub Issue #7)." % (candidate_flags,)\n        )\n    return config\n\n\ndef load_code_tokenizer(\n    tokenizer_name: str = DEFAULT_CODE_TOKENIZER,\n    hf_cache_dir: Optional[str] = None,\n    trust_remote_code: Optional[bool] = None,\n):\n    configure_huggingface_cache(hf_cache_dir)\n    if trust_remote_code is None:\n        trust_remote_code = _is_neobert_model(tokenizer_name)\n    return AutoTokenizer.from_pretrained(\n        tokenizer_name,\n        use_fast=True,\n        cache_dir=hf_cache_dir,\n        trust_remote_code=trust_remote_code,\n    )\n\n\ndef load_code_encoder(\n    model_name: str = DEFAULT_CODE_MODEL,\n    dtype_policy: str = "auto",\n    device: Optional[str] = None,\n    freeze: bool = True,\n    hf_cache_dir: Optional[str] = None,\n    trust_remote_code: Optional[bool] = None,\n) -> nn.Module:\n    device = device or ("cuda" if torch.cuda.is_available() else "cpu")\n    configure_huggingface_cache(hf_cache_dir)\n    dtype = _dtype_from_policy(dtype_policy, device)\n\n    is_neobert = _is_neobert_model(model_name)\n    if trust_remote_code is None:\n        trust_remote_code = is_neobert\n\n    kwargs: Dict[str, Any] = {"cache_dir": hf_cache_dir, "trust_remote_code": trust_remote_code}\n    if dtype is not None:\n        kwargs["torch_dtype"] = dtype\n\n    if is_neobert:\n        # Load + patch the config explicitly (rather than relying on\n        # AutoModel.from_pretrained\'s implicit config loading) so the\n        # unpadding override in _apply_neobert_config_overrides is guaranteed\n        # to be in effect before the backbone is instantiated.\n        config = AutoConfig.from_pretrained(\n            model_name, cache_dir=hf_cache_dir, trust_remote_code=trust_remote_code\n        )\n        config = _apply_neobert_config_overrides(config)\n        kwargs["config"] = config\n\n    model = AutoModel.from_pretrained(model_name, **kwargs)\n    model.to(device)\n\n    if freeze:\n        for param in model.parameters():\n            param.requires_grad = False\n        model.eval()\n\n    return model\n\n\ndef mean_pool_last_hidden(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\n    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)\n    summed = (last_hidden_state * mask).sum(dim=1)\n    denom = mask.sum(dim=1).clamp(min=1.0)\n    return summed / denom\n\n\ndef cls_pool_last_hidden(last_hidden_state: torch.Tensor) -> torch.Tensor:\n    return last_hidden_state[:, 0, :]\n\n\nclass CodeSequenceClassifier(nn.Module):\n    def __init__(\n        self,\n        model_name: str = DEFAULT_CODE_MODEL,\n        num_labels: int = 1,\n        freeze_backbone: bool = False,\n        pooling: str = "mean",\n        dtype_policy: str = "auto",\n        hf_cache_dir: Optional[str] = None,\n        trust_remote_code: Optional[bool] = None,\n        enforce_fp32_head: Optional[bool] = None,\n    ) -> None:\n        super().__init__()\n        device = "cuda" if torch.cuda.is_available() else "cpu"\n        self.backbone = load_code_encoder(\n            model_name=model_name,\n            dtype_policy=dtype_policy,\n            device=device,\n            freeze=freeze_backbone,\n            hf_cache_dir=hf_cache_dir,\n            trust_remote_code=trust_remote_code,\n        )\n        hidden_size = int(self.backbone.config.hidden_size)\n        self.classification_head = nn.Linear(hidden_size, num_labels)\n        self.pooling = pooling\n\n        # Mandatory Track-B safeguard (PDD sec. 5.2, GitHub Issue #11 on\n        # chandar-lab/NeoBERT -- NaN training bug): pooling + the\n        # classification head are forced to run in explicit float32,\n        # regardless of the ambient autocast dtype, so a bf16/fp16 NaN/Inf\n        # produced upstream in NeoBERT\'s attention stack doesn\'t get baked\n        # into the (trainable) head via a half-precision matmul. This is a\n        # wrapper-level mitigation -- it doesn\'t patch NeoBERT\'s own remote\n        # code, it just keeps *our* downstream math numerically safe.\n        if enforce_fp32_head is None:\n            enforce_fp32_head = _is_neobert_model(model_name)\n        self.enforce_fp32_head = enforce_fp32_head\n\n    @property\n    def config(self):\n        """Expose the underlying backbone config to pyreft/peft."""\n        return self.backbone.config\n\n    @property\n    def device(self) -> torch.device:\n        """Expose the device where parameters reside for pyreft."""\n        return next(self.parameters()).device\n\n    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, **kwargs: Any) -> torch.Tensor:\n        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask, **kwargs)\n        hidden = outputs.last_hidden_state\n\n        if self.enforce_fp32_head:\n            hidden = hidden.float()\n            attention_mask_for_pool = attention_mask.float()\n        else:\n            attention_mask_for_pool = attention_mask\n\n        if self.pooling == "cls":\n            pooled = cls_pool_last_hidden(hidden)\n        else:\n            pooled = mean_pool_last_hidden(hidden, attention_mask_for_pool)\n\n        if self.enforce_fp32_head:\n            # Disable autocast for the head matmul so it isn\'t silently\n            # downcast back to bf16/fp16 by the enclosing `torch.amp.autocast`\n            # context in the training loop.\n            with torch.autocast(device_type=pooled.device.type, enabled=False):\n                logits = self.classification_head(pooled.float())\n        else:\n            logits = self.classification_head(pooled)\n\n        if logits.ndim > 1 and logits.size(-1) == 1:\n            return logits.squeeze(-1)\n        return logits\n\n\ndef count_trainable_parameters(model: nn.Module) -> Dict[str, int]:\n    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    total = sum(p.numel() for p in model.parameters())\n    return {\n        "trainable_parameters": int(trainable),\n        "total_parameters": int(total),\n        "trainable_percent": float(100.0 * trainable / max(total, 1)),\n    }\n\n\ndef infer_lora_target_modules(model: nn.Module) -> List[str]:\n    module_names = [name for name, _ in model.named_modules()]\n    candidate_sets = [\n        ["qkv"],\n        ["q_proj", "v_proj"],\n        ["query", "value"],\n        ["in_proj"],\n    ]\n    for candidates in candidate_sets:\n        if all(any(name.endswith(candidate) or f".{candidate}" in name for name in module_names) for candidate in candidates):\n            return candidates\n    return ["query", "value"]\n\n\ndef create_lora_sequence_classifier(\n    model_name: str = DEFAULT_CODE_MODEL,\n    rank: int = 8,\n    lora_alpha: int = 16,\n    lora_dropout: float = 0.05,\n    pooling: str = "mean",\n    dtype_policy: str = "auto",\n    hf_cache_dir: Optional[str] = None,\n    trust_remote_code: Optional[bool] = None,\n):\n    from peft import LoraConfig, get_peft_model\n\n    base = CodeSequenceClassifier(\n        model_name=model_name,\n        freeze_backbone=False,\n        pooling=pooling,\n        dtype_policy=dtype_policy,\n        hf_cache_dir=hf_cache_dir,\n        trust_remote_code=trust_remote_code,\n    )\n    target_modules = infer_lora_target_modules(base)\n    config = LoraConfig(\n        r=rank,\n        lora_alpha=lora_alpha,\n        target_modules=target_modules,\n        lora_dropout=lora_dropout,\n        bias="none",\n        task_type="FEATURE_EXTRACTION",\n        modules_to_save=["classification_head"],\n    )\n    return get_peft_model(base, config)\n\n\ndef get_lora_model(\n    model_name: str = DEFAULT_CODE_MODEL,\n    rank: int = 8,\n    lora_alpha: int = 16,\n    pooling: str = "mean",\n    trust_remote_code: Optional[bool] = None,\n):\n    return create_lora_sequence_classifier(\n        model_name=model_name,\n        rank=rank,\n        lora_alpha=lora_alpha,\n        pooling=pooling,\n        trust_remote_code=trust_remote_code,\n    )\n\n\n# =====================================================================\n# EXP-5: HEFT (Hierarchical Efficient Fine-Tuning: LoRA -> freeze -> ReFT)\n# =====================================================================\n#\n# HEFT is a two-phase procedure:\n#   Phase 1 (LoRA): train a standard LoRA-adapted sequence classifier\n#                    (use get_lora_model() below, already defined above).\n#   Phase 2 (ReFT):  freeze *everything* learned in Phase 1 (LoRA adapters,\n#                    classification head, backbone) and train a LoReFT\n#                    intervention on top of the frozen, LoRA-adapted backbone\n#                    (use attach_reft_to_lora_model() below).\n#\n# These two builders are meant to be driven by the two-phase training loop in\n# exp5_heft.py -- they only construct models, they don\'t train anything.\n\ndef freeze_lora_parameters(model: nn.Module) -> None:\n    """Freezes LoRA adapter parameters learned in Phase 1."""\n    for name, param in model.named_parameters():\n        if "lora_" in name:\n            param.requires_grad = False\n\n\ndef reft_component_path(layer_target: int) -> str:\n    """\n    Dotted/bracket component path pyreft needs to locate the target layer\'s\n    output *inside a peft-wrapped model*.\n\n    peft.get_peft_model() re-nests the original module tree under\n    "base_model.model.*" rather than preserving the original top-level\n    attribute names. pyreft/pyvene resolve `component` strings via\n    nn.Module.get_submodule(), which walks the *real* module registry (not\n    Python attribute-forwarding), so a path like\n    "backbone.encoder.layer[i].output" -- valid on the bare, unwrapped model --\n    does not exist once the model has been wrapped with LoRA, and pyreft will\n    fail to find it. It must be prefixed with "base_model.model." to match the\n    wrapped model\'s actual module tree. (This mirrors the pattern pyreft\'s own\n    docs use for peft-wrapped causal LMs: "base_model.model.model.layers[i]...".)\n\n    NOTE: this hardcoded path assumes a BERT/RoBERTa-style module tree\n    (`encoder.layer[i].output`), which is what CodeBERTa (EXP-5) has. It does\n    NOT apply to NeoBERT (EXP-8) -- see resolve_reft_component_path below,\n    which is architecture-aware and used for both.\n    """\n    return f"base_model.model.backbone.encoder.layer[{layer_target}].output"\n\n\ndef resolve_reft_component_path(lora_model: nn.Module, layer_target: int, model_name: str) -> str:\n    """\n    Architecture-aware version of reft_component_path, used by\n    attach_reft_to_lora_model for both EXP-5 (CodeBERTa) and EXP-8 (NeoBERT).\n\n    CodeBERTa is a standard BERT/RoBERTa-style module tree\n    (`encoder.layer[i].output` per layer -- a dedicated submodule whose output\n    IS the post-residual hidden state), so the hardcoded reft_component_path\n    above is safe and is used as-is for backward compatibility with EXP-5.\n\n    NeoBERT\'s module tree is different and NOT part of the public\n    `transformers` library (it\'s loaded via `trust_remote_code`, and the\n    Hugging Face Hub copy of its modeling code is not pinned/inspectable\n    ahead of time the way a released `transformers` architecture is): its\n    encoder stack is `transformer_encoder` (an nn.ModuleList of\n    `EncoderBlock`s), and each block has no separate "output" submodule --\n    the residual add happens inline in EncoderBlock.forward(), so the block\n    itself (whose forward() return value IS the post-residual hidden state)\n    is the right interception point, not some child of it.\n\n    Rather than hardcode a guessed dotted path for NeoBERT (wrong by even one\n    level of nesting and pyreft fails after Phase-1 LoRA has already finished\n    training -- an expensive way to find out), this WALKS the model\'s actual\n    module registry via named_modules() and finds whichever module path ends\n    in "transformer_encoder.<layer_target>", so it\'s correct regardless of\n    exactly how many wrapper levels (peft\'s "base_model.model.", our own\n    "backbone.", NeoBERT\'s own internal "model." prefix, etc.) sit above it.\n    Raises immediately with a clear message (before any Phase-2 training\n    starts) if no matching module is found, rather than letting pyreft fail\n    with a more cryptic KeyError deeper inside get_reft_model().\n    """\n    if not _is_neobert_model(model_name):\n        return reft_component_path(layer_target)\n\n    suffix = f"transformer_encoder.{layer_target}"\n    matches = [name for name, _ in lora_model.named_modules() if name.endswith(suffix)]\n\n    if not matches:\n        available = sorted(\n            {name for name, _ in lora_model.named_modules() if "transformer_encoder" in name}\n        )\n        raise RuntimeError(\n            f"[reft] Could not find a module path ending in \'{suffix}\' in the "\n            f"LoRA-wrapped NeoBERT model\'s module tree (needed for EXP-8\'s ReFT phase). "\n            f"NeoBERT\'s Hub modeling code may have changed since this was written. "\n            f"Module paths containing \'transformer_encoder\' that WERE found: "\n            f"{available[:20]}{\'...\' if len(available) > 20 else \'\'}. "\n            f"Update resolve_reft_component_path in models.py to match the current structure."\n        )\n    if len(matches) > 1:\n        raise RuntimeError(\n            f"[reft] Ambiguous: found {len(matches)} module paths ending in \'{suffix}\' "\n            f"({matches}) -- expected exactly one. Refusing to guess which one is correct."\n        )\n    return matches[0]\n\n\ndef attach_reft_to_lora_model(\n    lora_model: nn.Module,\n    reft_rank: int = 4,\n    layer_target: int = 4,\n    freeze_previous_phase: bool = True,\n    model_name: str = DEFAULT_CODE_MODEL,\n):\n    """\n    HEFT Phase 2. Takes an already Phase-1-trained LoRA model (as returned by\n    get_lora_model / create_lora_sequence_classifier) and attaches a LoReFT\n    intervention on top of it.\n\n    `model_name` is used only to pick the right component-path resolution\n    strategy (see resolve_reft_component_path) -- it does not affect which\n    weights are loaded, since `lora_model` is already an instantiated model.\n\n    Freezing behaviour:\n      - `freeze_previous_phase=True` explicitly freezes the LoRA adapter\n        parameters first (belt-and-braces).\n      - pyreft.get_reft_model() *also* freezes every remaining parameter of\n        the wrapped model by design -- that\'s the whole point of ReFT: adapt\n        frozen representations via a small intervention instead of updating\n        weights. So after this call, the classification head and backbone end\n        up frozen too; only the newly added LoReFT intervention parameters\n        are trainable. That matches "train LoRA, freeze it, apply ReFT".\n    """\n    import pyreft\n\n    if freeze_previous_phase:\n        freeze_lora_parameters(lora_model)\n\n    hidden_size = int(lora_model.base_model.model.backbone.config.hidden_size)\n    component_path = resolve_reft_component_path(lora_model, layer_target, model_name)\n\n    # NOTE: pyreft.ReftConfig expects `representations` as plain dict(s), NOT a\n    # pyreft.RepresentationConfig object -- no such class exists in pyreft\'s\n    # public API. Every real example in pyreft\'s own README/docs builds it this way.\n    reft_config = pyreft.ReftConfig(\n        representations=[\n            {\n                "layer": layer_target,\n                "component": component_path,\n                "low_rank_dimension": reft_rank,\n                "intervention": pyreft.LoreftIntervention(\n                    embed_dim=hidden_size,\n                    low_rank_dimension=reft_rank,\n                ),\n            }\n        ]\n    )\n\n    # set_device=False prevents PyReft from probing custom module properties during init\n    heft_model = pyreft.get_reft_model(lora_model, reft_config, set_device=False)\n\n    return heft_model')
print("Wrote", "case_study_2/models.py")


Wrote case_study_2/models.py


In [12]:
(SRC_DIR / "case_study_2/data_loader.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/data_loader.py").write_text('from __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Optional, Dict, Any, List, Tuple\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import Dataset, DataLoader\n\n\nEMPTY_CODE_SENTINEL = "EMPTY_CODE_SAMPLE"\n\n\nclass CodeTextDataset(Dataset):\n    def __init__(\n        self,\n        dataframe: pd.DataFrame,\n        code_column: str = "normalized_code",\n        label_column: str = "label",\n        source_id_column: str = "source_row_id",\n        project_column: str = "project",\n    ) -> None:\n        self.df = dataframe.copy().reset_index(drop=True)\n        self.code_column = code_column\n        self.label_column = label_column\n        self.source_id_column = source_id_column\n        self.project_column = project_column\n\n        self.df[self.code_column] = self.df[self.code_column].fillna("").astype(str)\n        empty_mask = self.df[self.code_column].str.strip().eq("")\n        if empty_mask.any():\n            self.df.loc[empty_mask, self.code_column] = EMPTY_CODE_SENTINEL\n\n    def __len__(self) -> int:\n        return int(len(self.df))\n\n    def __getitem__(self, idx: int) -> Dict[str, Any]:\n        row = self.df.iloc[idx]\n        return {\n            "code": str(row[self.code_column]),\n            "label": int(row[self.label_column]),\n            "source_row_id": int(row[self.source_id_column]),\n            "project": str(row[self.project_column]),\n        }\n\n\n@dataclass\nclass TransformerBatchCollator:\n    tokenizer: Any\n    max_length: int = 512\n    pad_to_multiple_of: Optional[int] = 8\n\n    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:\n        texts = [feature["code"] for feature in features]\n        enc = self.tokenizer(\n            texts,\n            truncation=True,\n            max_length=self.max_length,\n            padding=True,\n            pad_to_multiple_of=self.pad_to_multiple_of,\n            return_tensors="pt",\n        )\n\n        labels = torch.tensor([feature["label"] for feature in features], dtype=torch.float32)\n        source_row_ids = torch.tensor([feature["source_row_id"] for feature in features], dtype=torch.long)\n        projects = [feature["project"] for feature in features]\n\n        enc["labels"] = labels\n        enc["label"] = labels\n        enc["source_row_id"] = source_row_ids\n        enc["project"] = projects\n        return enc\n\n\ndef create_dataloader(\n    dataframe: pd.DataFrame,\n    tokenizer: Any,\n    batch_size: int = 16,\n    max_length: int = 512,\n    shuffle: bool = False,\n    code_column: str = "normalized_code",\n    label_column: str = "label",\n    source_id_column: str = "source_row_id",\n    project_column: str = "project",\n    num_workers: int = 0,\n) -> DataLoader:\n    dataset = CodeTextDataset(\n        dataframe=dataframe,\n        code_column=code_column,\n        label_column=label_column,\n        source_id_column=source_id_column,\n        project_column=project_column,\n    )\n    collator = TransformerBatchCollator(\n        tokenizer=tokenizer,\n        max_length=max_length,\n        pad_to_multiple_of=8 if torch.cuda.is_available() else None,\n    )\n    return DataLoader(\n        dataset,\n        batch_size=batch_size,\n        shuffle=shuffle,\n        drop_last=False,\n        num_workers=num_workers,\n        pin_memory=torch.cuda.is_available(),\n        collate_fn=collator,\n    )\n\n\ndef get_pos_weight(dataframe: pd.DataFrame, label_column: str = "label") -> torch.Tensor:\n    y = dataframe[label_column].astype(int).values\n    neg = int((y == 0).sum())\n    pos = int((y == 1).sum())\n    if pos == 0:\n        return torch.tensor([1.0], dtype=torch.float32)\n    return torch.tensor([neg / pos], dtype=torch.float32)\n\n\ndef get_class_weights(dataframe: pd.DataFrame, label_column: str = "label") -> torch.Tensor:\n    return get_pos_weight(dataframe, label_column=label_column)\n\n\ndef sample_with_optional_positive_fraction(\n    frame: pd.DataFrame,\n    n_rows: int,\n    label_column: str = "label",\n    positive_fraction: Optional[float] = None,\n    random_state: int = 42,\n) -> pd.DataFrame:\n    if n_rows is None or n_rows <= 0 or len(frame) <= n_rows:\n        return frame.copy().reset_index(drop=True)\n\n    rng = np.random.default_rng(random_state)\n\n    if positive_fraction is None:\n        indices = rng.choice(frame.index.to_numpy(), size=n_rows, replace=False)\n        return frame.loc[indices].copy().reset_index(drop=True)\n\n    positives = frame[frame[label_column].astype(int) == 1]\n    negatives = frame[frame[label_column].astype(int) == 0]\n\n    n_pos = min(len(positives), max(1, int(round(n_rows * positive_fraction))))\n    n_neg = min(len(negatives), n_rows - n_pos)\n\n    pos_idx = rng.choice(positives.index.to_numpy(), size=n_pos, replace=False) if n_pos else []\n    neg_idx = rng.choice(negatives.index.to_numpy(), size=n_neg, replace=False) if n_neg else []\n    idx = np.concatenate([pos_idx, neg_idx])\n    rng.shuffle(idx)\n\n    return frame.loc[idx].copy().reset_index(drop=True)\n\n\ndef make_project_disjoint_threshold_split(\n    train_frame: pd.DataFrame,\n    threshold_fraction: float = 0.20,\n    project_column: str = "project",\n    label_column: str = "label",\n    random_state: int = 42,\n) -> Tuple[pd.DataFrame, pd.DataFrame]:\n    projects = train_frame[[project_column, label_column]].groupby(project_column)[label_column].agg(["count", "sum"])\n    project_names = projects.index.to_numpy()\n\n    rng = np.random.default_rng(random_state)\n    shuffled = project_names.copy()\n    rng.shuffle(shuffled)\n\n    target_rows = int(round(len(train_frame) * threshold_fraction))\n    selected = []\n    count = 0\n\n    for project in shuffled:\n        selected.append(project)\n        count += int(projects.loc[project, "count"])\n        if count >= target_rows:\n            break\n\n    selected = set(selected)\n    threshold_mask = train_frame[project_column].isin(selected)\n    threshold_frame = train_frame[threshold_mask].copy().reset_index(drop=True)\n    fit_frame = train_frame[~threshold_mask].copy().reset_index(drop=True)\n\n    if (\n        fit_frame[label_column].sum() == 0\n        or threshold_frame[label_column].sum() == 0\n        or len(fit_frame) == 0\n        or len(threshold_frame) == 0\n    ):\n        shuffled_rows = train_frame.sample(frac=1.0, random_state=random_state).reset_index(drop=True)\n        cut = max(1, int(round(len(shuffled_rows) * (1.0 - threshold_fraction))))\n        fit_frame = shuffled_rows.iloc[:cut].copy().reset_index(drop=True)\n        threshold_frame = shuffled_rows.iloc[cut:].copy().reset_index(drop=True)\n\n    return fit_frame, threshold_frame')
print("Wrote", "case_study_2/data_loader.py")


Wrote case_study_2/data_loader.py


In [13]:
(SRC_DIR / "case_study_2/exp6/__init__.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/exp6/__init__.py").write_text('')
print("Wrote", "case_study_2/exp6/__init__.py")


Wrote case_study_2/exp6/__init__.py


In [14]:
(SRC_DIR / "case_study_2/exp6/exp6_linear_probe.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/exp6/exp6_linear_probe.py").write_text('from __future__ import annotations\n\nimport gc\nimport json\nimport time\nfrom dataclasses import dataclass, asdict\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional, Tuple\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import average_precision_score, precision_recall_curve, confusion_matrix\nfrom sklearn.preprocessing import StandardScaler\nimport matplotlib.pyplot as plt\n\nfrom case_study_2.data_loader import create_dataloader\nfrom case_study_2.models import (\n    DEFAULT_NEOBERT_MODEL,\n    DEFAULT_NEOBERT_TOKENIZER,\n    configure_huggingface_cache,\n    load_code_tokenizer,\n    load_code_encoder,\n)\nfrom case_study_1 import evaluation\nfrom case_study_1 import split_manifest\nfrom case_study_1.evaluation import EvaluationConfig\nfrom case_study_1.confidence_intervals import bootstrap_metric_ci, format_ci_report\n\n\nEXP6_VERSION = "cs2-exp6-neobert-linear-probe-v1"\n\n\n@dataclass(frozen=True)\nclass Exp6Config:\n    experiment_name: str = "cs2_exp6_neobert_linear_probe"\n\n    code_column: str = "normalized_code"\n    source_id_column: str = "source_row_id"\n    label_column: str = "label"\n    project_column: str = "project"\n    fold_column: str = "fold"\n\n    model_name: str = DEFAULT_NEOBERT_MODEL\n    tokenizer_name: str = DEFAULT_NEOBERT_TOKENIZER\n    hf_cache_dir: Optional[str] = None\n    # NeoBERT supports up to 4096 tokens (RoPE/YaRE), unlike CodeBERTa\'s 512\n    # positional-embedding ceiling. Kept at 512 by default for apples-to-apples\n    # comparison with EXP-6/4/5/7 and because most DiverseVul functions are\n    # under 512 tokens anyway (see project discussion) -- raise this\n    # deliberately (and re-check VRAM/runtime) if you want to test whether\n    # the longer context recovers anything on the minority of longer\n    # functions that get truncated at 512.\n    max_length: int = 512\n    dtype_policy: str = "bfloat16"\n    embedding_batch_size: int = 64\n    pooling: str = "mean"\n    trust_remote_code: bool = True  # NeoBERT ships as trust_remote_code on the Hub\n\n    logistic_max_iter: int = 2000\n    logistic_solver: str = "lbfgs"\n    class_weight: str = "balanced"\n    decision_threshold: float = 0.50\n\n    n_splits: int = 5\n    random_state: int = 42\n    verbose: bool = True\n\n\n@dataclass(frozen=True)\nclass NestedProbeConfig:\n    experiment_name: str = "cs2_exp6_nested_probe_dev_grouped"\n    C_grid: Tuple[float, ...] = (1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0)\n    inner_n_splits: int = 3\n    inner_random_state: int = 20260707\n    selection_metric: str = "average_precision_pr_auc"\n    decision_threshold: float = 0.50\n    tie_break_rule: str = "higher_C_then_grid_order"\n    verbose: bool = True\n\n\ndef resolve_device(require_cuda: bool = True) -> str:\n    if torch.cuda.is_available():\n        return "cuda:0"\n    if require_cuda:\n        raise RuntimeError(\n            "EXP-6 linear probe requires a CUDA GPU for embedding extraction."\n        )\n    return "cpu"\n\n\n@torch.no_grad()\ndef extract_embeddings(\n    encoder,\n    tokenizer,\n    frame: pd.DataFrame,\n    config: Exp6Config,\n    device: str,\n    cache_path: Optional[Path] = None,\n    checkpoint_every: int = 100,\n) -> np.ndarray:\n    if cache_path is not None and cache_path.exists():\n        return np.load(cache_path)\n\n    if not str(device).startswith("cuda"):\n        raise RuntimeError("extract_embeddings must run on a CUDA device.")\n\n    frame = frame.reset_index(drop=True)\n    code_lengths = frame[config.code_column].fillna("").astype(str).str.len()\n    sort_order = code_lengths.sort_values(kind="mergesort").index.to_numpy()\n    sorted_frame = frame.iloc[sort_order].reset_index(drop=True)\n    inverse_order = np.argsort(sort_order)\n\n    n_rows = len(sorted_frame)\n    n_batches_total = -(-n_rows // config.embedding_batch_size)\n\n    checkpoint_path = cache_path.with_suffix(".checkpoint.npz") if cache_path is not None else None\n    all_embeddings: List[np.ndarray] = []\n    start_batch = 0\n\n    if checkpoint_path is not None and checkpoint_path.exists():\n        ckpt = np.load(checkpoint_path)\n        all_embeddings = [ckpt["embeddings"]]\n        start_batch = int(ckpt["n_batches"])\n\n    loader = create_dataloader(\n        sorted_frame,\n        tokenizer,\n        batch_size=config.embedding_batch_size,\n        max_length=config.max_length,\n        shuffle=False,\n        code_column=config.code_column,\n        label_column=config.label_column,\n        source_id_column=config.source_id_column,\n        project_column=config.project_column,\n        num_workers=2,\n    )\n\n    encoder.eval()\n    t0 = time.time()\n\n    for i, batch in enumerate(loader):\n        if i < start_batch:\n            continue\n\n        input_ids = batch["input_ids"].to(device, non_blocking=True)\n        attention_mask = batch["attention_mask"].to(device, non_blocking=True)\n\n        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):\n            outputs = encoder(input_ids=input_ids, attention_mask=attention_mask)\n            # --- Guardrail (PDD sec. 5.2 / chandar-lab/NeoBERT GitHub Issue #11) ---\n            # NeoBERT has a known numerical-instability bug that can produce\n            # NaN/Inf hidden states under bf16 in some configurations. The\n            # bug report is specifically about training, but a frozen\n            # forward-only pass in bf16 is not immune in principle, and this\n            # is cheap insurance: pooling arithmetic is done in fp32 (matching\n            # the enforce_fp32_head safeguard already used for NeoBERT\'s\n            # trainable head in CodeSequenceClassifier), and any NaN/Inf that\n            # does slip through fails loudly here instead of silently\n            # poisoning the linear probe\'s training data.\n            last_hidden = outputs.last_hidden_state.float()\n            if config.pooling == "cls":\n                pooled = last_hidden[:, 0, :]\n            else:\n                mask = attention_mask.unsqueeze(-1).to(last_hidden.dtype)\n                summed = (last_hidden * mask).sum(dim=1)\n                denom = mask.sum(dim=1).clamp(min=1.0)\n                pooled = summed / denom\n\n        if not torch.isfinite(pooled).all():\n            n_bad = (~torch.isfinite(pooled)).any(dim=-1).sum().item()\n            raise RuntimeError(\n                f"[embed] NaN/Inf detected in pooled embeddings for {n_bad}/{pooled.shape[0]} "\n                f"rows in batch {i} (see PDD sec. 5.2 / chandar-lab/NeoBERT GitHub Issue #11). "\n                f"This is the known NeoBERT numerical-instability bug, not a data problem -- "\n                f"do not silently drop/zero these rows, investigate the encoder/dtype config first."\n            )\n\n        all_embeddings.append(pooled.detach().cpu().numpy())\n\n        if checkpoint_path is not None and (i + 1) % checkpoint_every == 0:\n            partial = np.concatenate(all_embeddings, axis=0)\n            np.savez(checkpoint_path, embeddings=partial, n_batches=i + 1)\n            if config.verbose:\n                elapsed_min = (time.time() - t0) / 60\n                print(f"  [embed] checkpoint @ batch {i+1}/{n_batches_total} | elapsed {elapsed_min:.1f} min")\n\n    embeddings_sorted = np.concatenate(all_embeddings, axis=0)\n    embeddings = embeddings_sorted[inverse_order]\n\n    if cache_path is not None:\n        cache_path.parent.mkdir(parents=True, exist_ok=True)\n        np.save(cache_path, embeddings)\n        if checkpoint_path is not None and checkpoint_path.exists():\n            checkpoint_path.unlink()\n\n    return embeddings\n\n\ndef _fit_probe(X: np.ndarray, y: np.ndarray, C: float, config: Exp6Config) -> Tuple[StandardScaler, LogisticRegression]:\n    scaler = StandardScaler(copy=False)\n    X_s = scaler.fit_transform(X.astype(np.float32, copy=False))\n    clf = LogisticRegression(\n        C=C,\n        max_iter=config.logistic_max_iter,\n        solver=config.logistic_solver,\n        class_weight=config.class_weight,\n        random_state=config.random_state,\n    )\n    clf.fit(X_s, y)\n    return scaler, clf\n\n\ndef _predict_probe(scaler: StandardScaler, clf: LogisticRegression, X: np.ndarray) -> np.ndarray:\n    return clf.predict_proba(scaler.transform(X.astype(np.float32, copy=False)))[:, 1]\n\n\ndef _grouped_inner_folds(\n    frame: pd.DataFrame,\n    source_id_column: str,\n    label_column: str,\n    project_column: str,\n    n_splits: int,\n    random_state: int,\n) -> pd.DataFrame:\n    inner_split_config = split_manifest.SplitConfig(\n        n_splits=n_splits,\n        random_state=random_state,\n        shuffle=True,\n        source_id_column=source_id_column,\n        label_column=label_column,\n        group_column=project_column,\n    )\n    return split_manifest.create_project_grouped_manifest(\n        frame[[source_id_column, label_column, project_column]],\n        config=inner_split_config,\n    )\n\n\ndef run_exp6_nested_inner_profile(\n    development_frame: pd.DataFrame,\n    development_embeddings: np.ndarray,\n    development_manifest: pd.DataFrame,\n    outer_fold_id: int,\n    base_config: Exp6Config,\n    nested_config: NestedProbeConfig,\n) -> Dict[str, Any]:\n    t0 = time.time()\n\n    id_to_pos = {rid: pos for pos, rid in enumerate(development_frame[base_config.source_id_column].values)}\n\n    outer_train_ids = set(\n        development_manifest.loc[\n            development_manifest[base_config.fold_column] != outer_fold_id, base_config.source_id_column\n        ]\n    )\n    train_frame = development_frame[\n        development_frame[base_config.source_id_column].isin(outer_train_ids)\n    ].reset_index(drop=True)\n\n    inner_manifest = _grouped_inner_folds(\n        train_frame, base_config.source_id_column, base_config.label_column,\n        base_config.project_column, nested_config.inner_n_splits, nested_config.inner_random_state,\n    )\n\n    rows = []\n    for inner_id in range(nested_config.inner_n_splits):\n        tr_ids = set(inner_manifest.loc[inner_manifest["fold"] != inner_id, "source_row_id"])\n        va_ids = set(inner_manifest.loc[inner_manifest["fold"] == inner_id, "source_row_id"])\n        tr_frame = train_frame[train_frame[base_config.source_id_column].isin(tr_ids)]\n        va_frame = train_frame[train_frame[base_config.source_id_column].isin(va_ids)]\n        tr_pos = [id_to_pos[rid] for rid in tr_frame[base_config.source_id_column].values]\n        va_pos = [id_to_pos[rid] for rid in va_frame[base_config.source_id_column].values]\n\n        X_tr = development_embeddings[tr_pos]\n        y_tr = tr_frame[base_config.label_column].astype(int).values\n        X_va = development_embeddings[va_pos]\n        y_va = va_frame[base_config.label_column].astype(int).values\n\n        for C in nested_config.C_grid:\n            scaler, clf = _fit_probe(X_tr, y_tr, C, base_config)\n            scores = _predict_probe(scaler, clf, X_va)\n            ap = average_precision_score(y_va, scores) if len(np.unique(y_va)) > 1 else float("nan")\n            rows.append({"inner_fold": inner_id, "C": C, "average_precision_pr_auc": ap, "n_val": len(y_va)})\n            del scaler, clf, scores\n            gc.collect()\n\n        del tr_frame, va_frame, X_tr, X_va, y_tr, y_va\n        gc.collect()\n\n    alpha_summary = (\n        pd.DataFrame(rows)\n        .groupby("C", as_index=False)["average_precision_pr_auc"]\n        .mean()\n        .sort_values(["average_precision_pr_auc", "C"], ascending=[False, False])\n        .reset_index(drop=True)\n    )\n    selected_C = float(alpha_summary.iloc[0]["C"])\n\n    del train_frame\n    gc.collect()\n\n    return {\n        "outer_fold_id": outer_fold_id,\n        "selected_C": {"outer_fold_id": outer_fold_id, "selected_C": selected_C},\n        "C_summary": alpha_summary,\n        "inner_split_audit": pd.DataFrame(rows),\n        "total_profile_seconds": time.time() - t0,\n    }\n\n\ndef _checkpoint_paths(output_dir: Path, outer_fold_id: int) -> Dict[str, Path]:\n    root = output_dir / "checkpoints"\n    root.mkdir(parents=True, exist_ok=True)\n    prefix = f"outer_fold_{outer_fold_id}"\n    return {\n        "predictions": root / f"{prefix}_predictions.parquet",\n        "selected": root / f"{prefix}_selected_C.json",\n        "training": root / f"{prefix}_outer_training.json",\n    }\n\n\ndef _write_outer_checkpoint(output_dir: Path, outer_fold_id: int, predictions: pd.DataFrame, selected: dict, training: dict) -> None:\n    paths = _checkpoint_paths(output_dir, outer_fold_id)\n    predictions.to_parquet(paths["predictions"], index=False)\n    with paths["selected"].open("w", encoding="utf-8") as f:\n        json.dump(selected, f, indent=2, default=str)\n    with paths["training"].open("w", encoding="utf-8") as f:\n        json.dump(training, f, indent=2, default=str)\n\n\ndef _load_outer_checkpoint(output_dir: Path, outer_fold_id: int) -> Optional[dict]:\n    paths = _checkpoint_paths(output_dir, outer_fold_id)\n    if not all(p.exists() for p in paths.values()):\n        return None\n    with paths["selected"].open("r", encoding="utf-8") as f:\n        selected = json.load(f)\n    with paths["training"].open("r", encoding="utf-8") as f:\n        training = json.load(f)\n    return {\n        "predictions": pd.read_parquet(paths["predictions"]),\n        "selected": selected,\n        "training": training,\n    }\n\n\ndef _update_run_state(state_path: Path, completed_folds, status: str) -> None:\n    state = {\n        "status": status,\n        "updated_utc": datetime.now(timezone.utc).isoformat(),\n        "completed_outer_folds": sorted(int(f) for f in completed_folds),\n    }\n    with state_path.open("w", encoding="utf-8") as f:\n        json.dump(state, f, indent=2)\n\n\ndef run_exp6_nested_probe(\n    development_frame: pd.DataFrame,\n    development_embeddings: np.ndarray,\n    development_manifest: pd.DataFrame,\n    base_config: Exp6Config,\n    nested_config: NestedProbeConfig,\n    output_dir: Path,\n    additional_metadata: Optional[Dict[str, Any]] = None,\n    resume: bool = True,\n) -> Dict[str, Any]:\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    state_path = output_dir / "exp6_nested_run_state.json"\n    t0 = time.time()\n\n    id_to_pos = {rid: pos for pos, rid in enumerate(development_frame[base_config.source_id_column].values)}\n    fold_ids = sorted(development_manifest[base_config.fold_column].unique().tolist())\n\n    oof_parts = []\n    selected_rows = []\n    outer_training_rows = []\n    completed_folds = []\n\n    for outer_fold_id in fold_ids:\n        checkpoint = _load_outer_checkpoint(output_dir, outer_fold_id) if resume else None\n        if checkpoint is not None:\n            oof_parts.append(checkpoint["predictions"])\n            selected_rows.append(checkpoint["selected"])\n            outer_training_rows.append(checkpoint["training"])\n            completed_folds.append(outer_fold_id)\n            if nested_config.verbose:\n                print(f"  [nested] Outer fold {outer_fold_id}: loaded from checkpoint.")\n            continue\n\n        if nested_config.verbose:\n            print(f"  [nested] Outer fold {outer_fold_id}: inner C grid search...")\n\n        profile = run_exp6_nested_inner_profile(\n            development_frame, development_embeddings, development_manifest,\n            outer_fold_id, base_config, nested_config,\n        )\n        selected_C = profile["selected_C"]["selected_C"]\n\n        train_ids = set(\n            development_manifest.loc[\n                development_manifest[base_config.fold_column] != outer_fold_id, base_config.source_id_column\n            ]\n        )\n        val_ids = set(\n            development_manifest.loc[\n                development_manifest[base_config.fold_column] == outer_fold_id, base_config.source_id_column\n            ]\n        )\n        if train_ids.intersection(val_ids):\n            raise RuntimeError(f"Outer fold {outer_fold_id}: train/val ID leakage detected.")\n\n        train_frame = development_frame[development_frame[base_config.source_id_column].isin(train_ids)]\n        val_frame = development_frame[development_frame[base_config.source_id_column].isin(val_ids)]\n\n        tr_pos = [id_to_pos[rid] for rid in train_frame[base_config.source_id_column].values]\n        va_pos = [id_to_pos[rid] for rid in val_frame[base_config.source_id_column].values]\n\n        X_tr = development_embeddings[tr_pos]\n        y_tr = train_frame[base_config.label_column].astype(int).values\n        X_va = development_embeddings[va_pos]\n\n        scaler, clf = _fit_probe(X_tr, y_tr, selected_C, base_config)\n        val_scores = _predict_probe(scaler, clf, X_va)\n\n        fold_oof = pd.DataFrame({\n            base_config.source_id_column: val_frame[base_config.source_id_column].values,\n            base_config.project_column: val_frame[base_config.project_column].values,\n            "label": val_frame[base_config.label_column].astype(int).values,\n            "y_score": val_scores,\n            "fold": outer_fold_id,\n        })\n\n        selected_row = {"outer_fold_id": outer_fold_id, "selected_C": selected_C}\n        training_row = {\n            "outer_fold_id": outer_fold_id,\n            "selected_C": selected_C,\n            "n_train": int(len(train_frame)),\n            "n_val": int(len(val_frame)),\n            "train_projects": int(train_frame[base_config.project_column].nunique()),\n            "val_projects": int(val_frame[base_config.project_column].nunique()),\n        }\n\n        _write_outer_checkpoint(output_dir, outer_fold_id, fold_oof, selected_row, training_row)\n\n        oof_parts.append(fold_oof)\n        selected_rows.append(selected_row)\n        outer_training_rows.append(training_row)\n        completed_folds.append(outer_fold_id)\n\n        _update_run_state(state_path, completed_folds, status="running")\n\n        del train_frame, val_frame, X_tr, X_va, y_tr, scaler, clf, val_scores, profile\n        gc.collect()\n\n    oof_predictions = pd.concat(oof_parts, axis=0).reset_index(drop=True)\n\n    eval_config = EvaluationConfig(threshold=nested_config.decision_threshold, expected_n_folds=len(fold_ids))\n    eval_results = evaluation.evaluate_oof_predictions(oof_predictions, config=eval_config)\n\n    selected_df = pd.DataFrame(selected_rows)\n    outer_training_df = pd.DataFrame(outer_training_rows)\n\n    artifacts = {\n        "oof_predictions": output_dir / "exp6_nested_oof_predictions.parquet",\n        "selected_C_per_fold": output_dir / "exp6_selected_C_per_fold.csv",\n        "outer_training_audit": output_dir / "exp6_outer_training_audit.csv",\n        "run_metadata": output_dir / "exp6_nested_run_metadata.json",\n    }\n    oof_predictions.to_parquet(artifacts["oof_predictions"], index=False)\n    selected_df.to_csv(artifacts["selected_C_per_fold"], index=False)\n    outer_training_df.to_csv(artifacts["outer_training_audit"], index=False)\n\n    metadata = {\n        "exp6_version": EXP6_VERSION,\n        "base_config": asdict(base_config),\n        "nested_config": {**asdict(nested_config), "C_grid": list(nested_config.C_grid)},\n        "runtime_seconds": time.time() - t0,\n        **(additional_metadata or {}),\n    }\n    with open(artifacts["run_metadata"], "w", encoding="utf-8") as f:\n        json.dump(metadata, f, indent=2, default=str)\n\n    _update_run_state(state_path, completed_folds, status="completed")\n\n    return {\n        "oof_predictions": oof_predictions,\n        "evaluation": eval_results,\n        "selected_C": selected_df,\n        "outer_fold_training": outer_training_df,\n        "artifacts": artifacts,\n    }\n\n\ndef run_exp6_canonical_retrain(\n    development_frame: pd.DataFrame,\n    development_embeddings: np.ndarray,\n    selected_C: float,\n    base_config: Exp6Config,\n    output_dir: Path,\n) -> Tuple[StandardScaler, LogisticRegression]:\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    y_dev = development_frame[base_config.label_column].astype(int).values\n    scaler, clf = _fit_probe(development_embeddings, y_dev, selected_C, base_config)\n\n    joblib.dump(clf, output_dir / "final_exp6_linear_probe_model.joblib")\n    joblib.dump(scaler, output_dir / "final_exp6_scaler.joblib")\n\n    return scaler, clf\n\n\ndef run_exp6_holdout_evaluation(\n    holdout_frame: pd.DataFrame,\n    holdout_embeddings: np.ndarray,\n    scaler: StandardScaler,\n    clf: LogisticRegression,\n    base_config: Exp6Config,\n    output_dir: Path,\n    n_bootstrap: int = 1000,\n    confidence: float = 0.95,\n) -> Dict[str, Any]:\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    y_holdout = holdout_frame[base_config.label_column].astype(int).values\n    y_scores = _predict_probe(scaler, clf, holdout_embeddings)\n\n    holdout_predictions = pd.DataFrame({\n        base_config.source_id_column: holdout_frame[base_config.source_id_column].values,\n        "project": holdout_frame[base_config.project_column].values,\n        "label": y_holdout,\n        "y_score": y_scores,\n        "fold": 0,\n    })\n    holdout_predictions.to_csv(output_dir / "exp6_holdout_predictions.csv", index=False)\n\n    eval_config = EvaluationConfig(threshold=base_config.decision_threshold, expected_n_folds=1)\n    holdout_metrics = evaluation.evaluate_oof_predictions(holdout_predictions, config=eval_config)\n    print(evaluation.format_metric_report(holdout_metrics["pooled_metrics"]))\n\n    ci_result = bootstrap_metric_ci(\n        holdout_predictions,\n        metric="average_precision_pr_auc",\n        group_column="project",\n        n_bootstrap=n_bootstrap,\n        confidence=confidence,\n        random_state=base_config.random_state,\n    )\n    with open(output_dir / "exp6_holdout_pr_auc_bootstrap_ci.json", "w", encoding="utf-8") as f:\n        json.dump(ci_result.as_dict(), f, indent=2)\n    print(format_ci_report(ci_result))\n\n    precision, recall, _ = precision_recall_curve(y_holdout, y_scores)\n    ap = holdout_metrics["pooled_metrics"]["average_precision_pr_auc"]\n    plt.figure(figsize=(6, 5))\n    plt.plot(recall, precision, color="b", label=f"EXP-6 Linear Probe (PR-AUC = {ap:.4f})")\n    plt.xlabel("Recall")\n    plt.ylabel("Precision")\n    plt.title("Precision-Recall Curve - Frozen Outer Holdout")\n    plt.legend(loc="lower left")\n    plt.grid(True)\n    plt.savefig(output_dir / "exp6_outer_holdout_pr_curve.png")\n    plt.close()\n\n    y_pred = (y_scores >= base_config.decision_threshold).astype(int)\n    cm = confusion_matrix(y_holdout, y_pred)\n    plt.figure(figsize=(4, 4))\n    plt.imshow(cm, cmap=plt.cm.Blues)\n    plt.title("Confusion Matrix - Frozen Outer Holdout")\n    plt.xlabel("Predicted")\n    plt.ylabel("Actual")\n    plt.xticks([0, 1], ["Non-Vuln (0)", "Vuln (1)"])\n    plt.yticks([0, 1], ["Non-Vuln (0)", "Vuln (1)"])\n    for i in range(2):\n        for j in range(2):\n            plt.text(j, i, str(cm[i, j]), ha="center", va="center")\n    plt.tight_layout()\n    plt.savefig(output_dir / "exp6_outer_holdout_confusion_matrix.png")\n    plt.close()\n\n    return {\n        "holdout_predictions": holdout_predictions,\n        "holdout_metrics": holdout_metrics,\n        "bootstrap_ci": ci_result.as_dict(),\n        "y_pred": y_pred,\n    }')
print("Wrote", "case_study_2/exp6/exp6_linear_probe.py")


Wrote case_study_2/exp6/exp6_linear_probe.py


## 6. Import project modules

In [15]:
import sys

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("case_study_2") or mod_name.startswith("case_study_1"):
        del sys.modules[mod_name]

from case_study_2.data_loader import create_dataloader
from case_study_2.models import (
    configure_huggingface_cache, load_code_tokenizer, load_code_encoder,
    DEFAULT_NEOBERT_MODEL, DEFAULT_NEOBERT_TOKENIZER,
)
from case_study_2.exp6.exp6_linear_probe import (
    Exp6Config, NestedProbeConfig, extract_embeddings,
    run_exp6_nested_inner_profile, run_exp6_nested_probe,
    run_exp6_canonical_retrain, run_exp6_holdout_evaluation,
)
from case_study_1 import split_manifest
from case_study_1 import evaluation
from case_study_1.confidence_intervals import bootstrap_metric_ci, paired_bootstrap_metric_ci, format_ci_report, format_paired_ci_report

print("Imported. Model:", DEFAULT_NEOBERT_MODEL)


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imported. Model: chandar-lab/NeoBERT


## 7. Load dataset and frozen manifests

In [16]:
import pandas as pd

full_df = pd.read_parquet(NORMALIZED_PARQUET)
outer_manifest_df = pd.read_parquet(OUTER_MANIFEST_PATH)
inner_manifest_df = pd.read_parquet(INNER_MANIFEST_PATH)

print("full_df rows:", len(full_df))
print("outer_manifest_df rows:", len(outer_manifest_df))
print("inner_manifest_df rows:", len(inner_manifest_df))


full_df rows: 261667
outer_manifest_df rows: 261667
inner_manifest_df rows: 203958


## 8. Build development and holdout frames

In [17]:
required_columns = {"source_row_id", "normalized_code", "label", "project"}
missing_columns = required_columns - set(full_df.columns)
if missing_columns:
    raise ValueError(f"Missing columns in full_df: {missing_columns}")

full_indexed = full_df.set_index("source_row_id", drop=False)
dev_ids = set(outer_manifest_df.loc[outer_manifest_df["partition"] == "development", "source_row_id"].tolist())
holdout_ids = set(outer_manifest_df.loc[outer_manifest_df["partition"] == "outer_holdout", "source_row_id"].tolist())
inner_ids = set(inner_manifest_df["source_row_id"].tolist())

if dev_ids != inner_ids:
    raise RuntimeError("Development partition and inner manifest coverage do not match.")
if dev_ids.intersection(holdout_ids):
    raise RuntimeError("Development and holdout partitions overlap.")

development_frame = full_indexed.loc[full_indexed["source_row_id"].isin(dev_ids)].reset_index(drop=True)
holdout_frame = full_indexed.loc[full_indexed["source_row_id"].isin(holdout_ids)].reset_index(drop=True)

print("development_frame rows:", len(development_frame))
print("holdout_frame rows:", len(holdout_frame))


development_frame rows: 203958
holdout_frame rows: 57709


## 9. Configuration objects

In [ ]:
base_config = Exp6Config(hf_cache_dir=HF_CACHE_DIR, code_column=CODE_COLUMN)
nested_config = NestedProbeConfig(C_grid=C_GRID)

print(base_config)
print(nested_config)


## 10. Smoke test on a small subsample

NeoBERT ships as `trust_remote_code` and has a known numerical-instability bug (PDD sec. 5.2 / GitHub Issue #11). Cheap sanity check on ~300 rows -- tokenizer, encoder loading, embedding extraction (with the NaN guardrail), and a quick probe fit -- before committing to the full development+holdout embedding extraction.

In [19]:
if RUN_SMOKE_TEST:
    smoke_df = development_frame.sample(n=min(300, len(development_frame)), random_state=42).reset_index(drop=True)

    print("Loading tokenizer...")
    _smoke_tokenizer = load_code_tokenizer(base_config.tokenizer_name, hf_cache_dir=HF_CACHE_DIR)
    print("Loading encoder...")
    _smoke_encoder = load_code_encoder(
        base_config.model_name, dtype_policy=base_config.dtype_policy,
        device=DEVICE, freeze=True, hf_cache_dir=HF_CACHE_DIR,
    )

    _smoke_embeddings = extract_embeddings(
        _smoke_encoder, _smoke_tokenizer, smoke_df, base_config, DEVICE,
        cache_path=None,
    )
    print("Smoke embeddings shape:", _smoke_embeddings.shape)
    assert _smoke_embeddings.shape[0] == len(smoke_df)
    import numpy as np
    assert np.isfinite(_smoke_embeddings).all(), "Non-finite values slipped past the guardrail -- investigate before proceeding."

    print("Smoke test passed: encoder loads, tokenizes, extracts finite embeddings.")

    del _smoke_encoder, _smoke_tokenizer, _smoke_embeddings
    torch.cuda.empty_cache()
else:
    print("RUN_SMOKE_TEST=False; skipping.")


Loading tokenizer...
Loading encoder...


A new version of the following files was downloaded from https://huggingface.co/chandar-lab/NeoBERT:
- rotary.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
2026-08-11 19:05:35.803612: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-11 19:05:35.817834: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1786475135.834539      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786475135.839466      55 cuda_blas.cc:1418] U

Smoke embeddings shape: (300, 768)
Smoke test passed: encoder loads, tokenizes, extracts finite embeddings.


## 11. Extract frozen NeoBERT embeddings

Runs the frozen encoder once over development and holdout, cached to disk so later cells never touch the tokenizer/encoder again.

In [20]:
try:
    print('Loading tokenizer...')
    tokenizer = load_code_tokenizer(base_config.tokenizer_name, hf_cache_dir=HF_CACHE_DIR)
    print('✓ Tokenizer loaded')

    print('Loading encoder...')
    encoder = load_code_encoder(
        base_config.model_name,
        dtype_policy=base_config.dtype_policy,
        device=DEVICE,
        freeze=True,
        hf_cache_dir=HF_CACHE_DIR,
    )
    print('✓ Encoder loaded successfully on', DEVICE)

    print('\nExtracting development embeddings...')
    development_embeddings = extract_embeddings(
        encoder, tokenizer, development_frame, base_config, DEVICE,
        cache_path=EMBEDDING_CACHE_DIR / 'development_embeddings.npy',
    )
    print('✓ Development embeddings extracted:', development_embeddings.shape)

    print('\nExtracting holdout embeddings...')
    holdout_embeddings = extract_embeddings(
        encoder, tokenizer, holdout_frame, base_config, DEVICE,
        cache_path=EMBEDDING_CACHE_DIR / 'holdout_embeddings.npy',
    )
    print('✓ Holdout embeddings extracted:', holdout_embeddings.shape)

    del encoder
    torch.cuda.empty_cache()
    print('✓ VRAM cleaned up')

except RuntimeError as e:
    if "NaN/Inf detected" in str(e):
        print('\n' + '='*70)
        print('ERROR: NeoBERT numerical-instability bug triggered (PDD sec. 5.2 / Issue #11)')
        print('='*70)
        print(str(e))
        print('\nDo not silently drop/zero these rows. Investigate dtype_policy '
              '(try dtype_policy=\'float32\' on base_config) before re-running.')
    raise
except Exception as e:
    print(f'ERROR during embedding extraction: {type(e).__name__}: {e}')
    import traceback
    traceback.print_exc()
    raise


Loading tokenizer...
✓ Tokenizer loaded
Loading encoder...


/workspace/DiverseVul--IS-Project/vuln-detection/src/case_study_2/models.py:91: UserWarning: [models] Could not find a known unpadding flag on the NeoBERT config (checked: ('use_unpadding', 'unpad_inputs', 'unpad', 'pack_sequences')). This revision of chandar-lab/NeoBERT may handle padding differently -- double check attention-mask correctness manually (see PDD sec. 5.2 / GitHub Issue #7).
  warnings.warn(


✓ Encoder loaded successfully on cuda:0

Extracting development embeddings...
  [embed] checkpoint @ batch 100/3187 | elapsed 0.0 min
  [embed] checkpoint @ batch 200/3187 | elapsed 0.1 min
  [embed] checkpoint @ batch 300/3187 | elapsed 0.1 min
  [embed] checkpoint @ batch 400/3187 | elapsed 0.2 min
  [embed] checkpoint @ batch 500/3187 | elapsed 0.2 min
  [embed] checkpoint @ batch 600/3187 | elapsed 0.3 min
  [embed] checkpoint @ batch 700/3187 | elapsed 0.3 min
  [embed] checkpoint @ batch 800/3187 | elapsed 0.4 min
  [embed] checkpoint @ batch 900/3187 | elapsed 0.5 min
  [embed] checkpoint @ batch 1000/3187 | elapsed 0.6 min
  [embed] checkpoint @ batch 1100/3187 | elapsed 0.7 min
  [embed] checkpoint @ batch 1200/3187 | elapsed 0.8 min
  [embed] checkpoint @ batch 1300/3187 | elapsed 0.9 min
  [embed] checkpoint @ batch 1400/3187 | elapsed 1.0 min
  [embed] checkpoint @ batch 1500/3187 | elapsed 1.2 min
  [embed] checkpoint @ batch 1600/3187 | elapsed 1.3 min
  [embed] checkpoin

## 12. Single-fold inner profile

Feasibility check on one outer-development fold before committing to the full nested run.

In [21]:
if RUN_PROFILE:
    profile = run_exp6_nested_inner_profile(
        development_frame=development_frame,
        development_embeddings=development_embeddings,
        development_manifest=inner_manifest_df,
        outer_fold_id=4,
        base_config=base_config,
        nested_config=nested_config,
    )
    print("Profile duration minutes:", profile["total_profile_seconds"] / 60)
    display(pd.DataFrame([profile["selected_C"]]))
    display(profile["C_summary"])
else:
    print("RUN_PROFILE=False; skipping.")


Profile duration minutes: 9.564959466457367


,outer_fold_id,selected_C
0,4,0.0001


,C,average_precision_pr_auc
0,0.0001,0.106895
1,0.0010,0.102363
2,0.0100,0.096836
3,1.0000,0.095043
4,0.1000,0.095028
5,10.0000,0.094977


## 13. Official nested probe

Checkpointed per outer fold on disk; safe to re-run after an interrupted session.

In [22]:
if RUN_NESTED_OFFICIAL:
    nested_results = run_exp6_nested_probe(
        development_frame=development_frame,
        development_embeddings=development_embeddings,
        development_manifest=inner_manifest_df,
        base_config=base_config,
        nested_config=nested_config,
        output_dir=EXP6_OUTPUT_DIR,
        additional_metadata={
            "input_parquet": str(NORMALIZED_PARQUET),
            "outer_manifest_path": str(OUTER_MANIFEST_PATH),
            "inner_manifest_path": str(INNER_MANIFEST_PATH),
        },
    )
    display(nested_results["selected_C"])
    print("Pooled nested PR-AUC:", nested_results["evaluation"]["pooled_metrics"]["average_precision_pr_auc"])
else:
    nested_results = None
    print("RUN_NESTED_OFFICIAL=False; skipping.")


  [nested] Outer fold 0: inner C grid search...
  [nested] Outer fold 1: inner C grid search...
  [nested] Outer fold 2: inner C grid search...
  [nested] Outer fold 3: inner C grid search...
  [nested] Outer fold 4: inner C grid search...


,outer_fold_id,selected_C
0,0,0.0001
1,1,0.0001
2,2,0.0001
3,3,0.0001
4,4,0.0001


Pooled nested PR-AUC: 0.11134743086477268


## 14. Canonical retrain on full development set

In [23]:
if RUN_CANONICAL_RETRAIN and nested_results is not None:
    global_selected_C = float(nested_results["selected_C"]["selected_C"].mode()[0])
    final_scaler, final_clf = run_exp6_canonical_retrain(
        development_frame=development_frame,
        development_embeddings=development_embeddings,
        selected_C=global_selected_C,
        base_config=base_config,
        output_dir=EXP6_OUTPUT_DIR,
    )
    print("Canonical model trained with C =", global_selected_C)
else:
    final_scaler, final_clf = None, None
    print("RUN_CANONICAL_RETRAIN=False or no nested results; skipping.")


Canonical model trained with C = 0.0001


## 15. Frozen outer holdout evaluation with bootstrap confidence interval

In [24]:
if RUN_HOLDOUT_EVAL and final_clf is not None:
    holdout_results = run_exp6_holdout_evaluation(
        holdout_frame=holdout_frame,
        holdout_embeddings=holdout_embeddings,
        scaler=final_scaler,
        clf=final_clf,
        base_config=base_config,
        output_dir=EXP6_OUTPUT_DIR,
    )
else:
    holdout_results = None
    print("RUN_HOLDOUT_EVAL=False or no canonical model available; skipping.")


Pooled Out-of-Fold Evaluation
                   n_samples: 57709
                vulnerable_1: 3211
            non_vulnerable_0: 54498
               positive_rate: 0.055641
                   threshold: 0.500000
    average_precision_pr_auc: 0.120369
                   precision: 0.096912
                      recall: 0.669573
                          f1: 0.169318
                         mcc: 0.142281
                 specificity: 0.632372
         false_positive_rate: 0.367628
               true_negative: 34463
              false_positive: 20035
              false_negative: 1061
               true_positive: 2150
average_precision_pr_auc: point estimate = 0.1204
  95% CI (project-block bootstrap): [0.1037, 0.1413]
  valid resamples: 1000/1000 (0 degenerate, dropped)
  n_projects: 203, random_state=42
  Reflects sampling variability within this dataset only; not an estimate of generalization to C functions outside this collection.


## 16. Paired comparisons on the frozen holdout

Two comparisons, both meaningful for different reasons:
- **EXP-6 vs EXP-3**: same method (frozen linear probe), different backbone -- isolates the effect of NeoBERT vs CodeBERTa on the frozen representation alone.
- **EXP-6 vs EXP-7**: same backbone (NeoBERT), different method -- isolates whether fine-tuning (LoRA) helps over the frozen probe, mirroring the EXP-3 vs EXP-4 comparison already run.

In [ ]:
EXP3_HOLDOUT_PREDICTIONS_PATH = OUTPUT_ROOT / "case_study_2" / f"exp3_codeberta_linear_probe_v1_{CODE_COLUMN_TAG}" / "exp3_holdout_predictions.csv"
EXP7_HOLDOUT_PREDICTIONS_PATH = OUTPUT_ROOT / "case_study_2" / f"exp7_neobert_lora_v1_{CODE_COLUMN_TAG}" / "exp7_holdout_predictions.csv"

if holdout_results is not None and EXP3_HOLDOUT_PREDICTIONS_PATH.exists():
    exp3_holdout = pd.read_csv(EXP3_HOLDOUT_PREDICTIONS_PATH)
    comparison_backbone = paired_bootstrap_metric_ci(
        predictions_a=holdout_results["holdout_predictions"],
        predictions_b=exp3_holdout,
        experiment_name_a="EXP-6 NeoBERT linear probe",
        experiment_name_b="EXP-3 CodeBERTa linear probe",
        metric="average_precision_pr_auc",
    )
    print(format_paired_ci_report(comparison_backbone))
else:
    print("Run EXP-6 holdout evaluation and the EXP-3 notebook first (backbone comparison).")

print()

if holdout_results is not None and EXP7_HOLDOUT_PREDICTIONS_PATH.exists():
    exp7_holdout = pd.read_csv(EXP7_HOLDOUT_PREDICTIONS_PATH)
    comparison_method = paired_bootstrap_metric_ci(
        predictions_a=holdout_results["holdout_predictions"],
        predictions_b=exp7_holdout,
        experiment_name_a="EXP-6 NeoBERT linear probe",
        experiment_name_b="EXP-7 NeoBERT LoRA",
        metric="average_precision_pr_auc",
    )
    print(format_paired_ci_report(comparison_method))
else:
    print("Run EXP-6 holdout evaluation and the EXP-7 notebook first (method comparison).")


## 17. Cleanup

In [26]:
import gc
import shutil
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("VRAM allocated:", torch.cuda.memory_allocated() / 1e9, "GB")

total, used, free = shutil.disk_usage(WORKSPACE_ROOT)
print(f"Disk usage at {WORKSPACE_ROOT}: {used/1e9:.1f} GB used / {total/1e9:.1f} GB total ({free/1e9:.1f} GB free)")
if used / 1e9 > STORAGE_CAP_GB:
    print(f"WARNING: workspace usage exceeds the {STORAGE_CAP_GB} GB storage cap -- consider pruning old checkpoints under {EXP6_OUTPUT_DIR}.")


VRAM allocated: 0.00851968 GB
Disk usage at /workspace: 5235.7 GB used / 5714.2 GB total (190.4 GB free)
